<a href="https://colab.research.google.com/github/aleksandrovd2-dev/ACAS_methodology/blob/main/ACAS_python_library_(rus).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import math
from dataclasses import dataclass
from typing import Tuple

@dataclass
class ACASConfig:
    """
    Глобальные настройки и константы модели ACAS.
    """
    R1: float = 0.5       # Фиксированный стартовый рейтинг новичка
    PM: float = 3.0       # Точка медианного созревания (месяцы)
    k: float = 2.0        # Коэффициент крутизны перехода функции Хилла
    MAX_PL: float = 12.0  # Ограничение скользящего окна в месяцах

class ACASScorer:
    """
    Основной класс для расчета ACAS.
    """
    def __init__(self, config: ACASConfig = ACASConfig()):
        self.config = config

    def calculate(
        self,
        pl: float,
        pa: float,
        vi: float,
        vp: float,
        vr: float,
        ai: float,
        ap: float,
        ar: float
    ) -> Tuple[float, str]:
        """
        Производит расчет рейтинга клиента и возвращает % и текстовый статус.

        Параметры:
        pl (Period of Life) - Период жизни клиента (мес)
        pa (Period of Activity) - Периоды активности (мес)
        vi (Volume of Invoices) - Кол-во выставленных счетов
        vp (Volume of Paid) - Кол-во оплаченных счетов
        vr (Volume of Refunds) - Кол-во возвратов
        ai (Amount Invoiced) - Сумма выставленных счетов
        ap (Amount Paid) - Сумма оплат
        ar (Amount Refunded) - Сумма возвратов
        """

        # 1. Ограничение скользящего окна (Rolling Window)
        # Защита от нулевого PL для предотвращения деления на ноль
        safe_pl = max(min(pl, self.config.MAX_PL), 0.1)
        safe_pa = max(pa, 0.1)

        # 2. Взвешенная эффективность (WE) [cite: 39]
        # Защита от деления на ноль, если счетов нет
        if vi == 0 or ai == 0:
            we = 0.0
        else:
            we = 0.5 * (((vp - vr) / vi) + ((ap - ar) / ai))
            we = max(0.0, min(we, 1.0)) # Метрика строго лежит в диапазоне от 0 до 1

        # 3. Дефицит активности (AG)
        ag = 1.0 - (safe_pa / safe_pl)
        ag = max(0.0, ag) # Защита от отрицательных значений, если PA > PL из-за ошибок в данных

        # 4. Коэффициент нулевого давления (ZP)
        # Использует количественные метрики документооборота
        zp = (vi - vp + vr) / safe_pa
        zp = max(0.0, zp)

        # 5. Индекс потерь (WI)
        wi = 0.5 * (ag + zp)

        # 6. Жесткий расчетный рейтинг (R2)
        # Если WI > 1 (огромное нулевое давление), рейтинг опускается до 0
        penalty = max(0.0, 1.0 - wi)
        r2 = we * penalty

        # 7. Динамическое сглаживание (w_pl) - Адаптированная функция Хилла
        w_pl = 1.0 / (1.0 + (safe_pl / self.config.PM) ** self.config.k)

        # 8. Итоговый синергетический рейтинг (Racas)
        racas_score = w_pl * self.config.R1 + (1.0 - w_pl) * r2

        # Перевод в проценты
        racas_percent = round(racas_score * 100, 2)

        # 9. Определение статуса по матрице
        status = self._get_status(racas_percent)

        return racas_percent, status

    def _get_status(self, score: float) -> str:
        """
        Переводит процентный рейтинг в бизнес-статус.
        """
        if score < 0.0:
            return "Критический"
        elif 0.0 == score <= 0.99:
            return "Нулевой"
        elif 1.0 <= score <= 5.0:
            return "Очень низкий"
        elif 5.01 <= score <= 25.0:
            return "Низкий"
        elif 25.01 <= score <= 65.0:
            return "Средний"
        elif 65.01 <= score <= 95.0:
            return "Высокий"
        else:
            return "Очень высокий"

# ==========================================
# Блок тестирования на примерах из статьи
# ==========================================
if __name__ == "__main__":
    scorer = ACASScorer()

    print("=== Тестирование методологии ACAS ===")

    # Клиент №1: Перспективный новичок [cite: 85, 86, 87, 88]
    # Примечание: В коде используется строгая формула ZP = (VI - VP + VR) / PA.
    score1, status1 = scorer.calculate(
        pl=1.5, pa=1.0,
        vi=2.0, vp=2.0, vr=0.0,
        ai=1.0, ap=0.9, ar=0.0
    )
    print(f"Клиент №1 (Новичок): Рейтинг {score1}% | Статус: {status1}")

    # Клиент №2: Старый «выжигатель времени» [cite: 105, 106, 107]
    score2, status2 = scorer.calculate(
        pl=12.0, pa=2.0,
        vi=50.0, vp=2.0, vr=0.0,
        ai=1.0, ap=0.2, ar=0.0
    )
    print(f"Клиент №2 (Пассивный): Рейтинг {score2}% | Статус: {status2}")

    # Клиент №3: Надежный постоянный партнер [cite: 127, 128, 129]
    score3, status3 = scorer.calculate(
        pl=6.0, pa=6.0,
        vi=12.0, vp=12.0, vr=0.0,
        ai=1.0, ap=1.0, ar=0.0
    )
    print(f"Клиент №3 (Надежный): Рейтинг {score3}% | Статус: {status3}")

=== Тестирование методологии ACAS ===
Клиент №1 (Новичок): Рейтинг 55.83% | Статус: Средний
Клиент №2 (Пассивный): Рейтинг 2.94% | Статус: Очень низкий
Клиент №3 (Надежный): Рейтинг 90.0% | Статус: Высокий
